# Week 3, Notebook 1: Money and Statistics

**Reading your own money like a data scientist.**

*Author: The Genius Project Year 3*

---

### What you will build
- A clear picture of where your money comes from and where it goes.
- The core numbers every analyst starts with: average, middle, and spread.
- Charts that reveal the hidden patterns in a year of spending.

### The big idea, in one breath
Money leaves a trail of numbers. If you learn to read those numbers, you can see
your habits, spot the surprises, and make better choices. Everything else this
week &mdash; forecasting, simulation, machine learning &mdash; is built on top of
these first ideas.

> This week has no football. It uses a made-up but realistic set of money data so
> you can learn the tools that banks, budgeting apps, and trading desks use for real.

### The data, in plain words

Everything lives in the `data` folder. The first file is `monthly_budget.csv`:
**48 months** (four years) of one teenager's money, one row per month.

| Column | What it means |
| --- | --- |
| `month` | The month, written as year-month, for example `2024-07`. |
| `allowance` | Money given by family each month. |
| `side_job` | Money earned from work, like tutoring or a weekend shift. |
| `gifts` | One-off money, such as a birthday or holiday gift. |
| `income` | All the money that came in that month (the three above added up). |
| `food` | Spent on snacks, drinks, and eating out. |
| `transport` | Spent on the bus, train, or fuel. |
| `entertainment` | Spent on games, cinema, and going out. |
| `clothes` | Spent on clothes and shoes. |
| `phone` | A fixed monthly phone plan. |
| `other` | Anything that does not fit the boxes above. |
| `spending` | All the money that went out that month (the six categories added up). |
| `saved` | Income minus spending. Can be negative in a bad month. |
| `balance` | The running total in the savings jar after that month. |
| `over_budget` | `1` if you spent more than you earned that month, else `0`. |

### Step 1: Load the data and take a first look

Every project starts here. We load the table and look at the first few rows to
check it came in cleanly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the 48 months of money data.
budget = pd.read_csv("data/monthly_budget.csv")

print("Rows (months):", len(budget))
print("Columns:", list(budget.columns))
budget.head()

### Step 2: The three numbers analysts always start with

Before any fancy model, you describe the data. Three numbers do most of the work:

- **Mean (the average):** add everything up, divide by how many. The typical value.
- **Median (the middle):** line the values up and take the one in the centre.
- **Standard deviation (the spread):** how far, on average, months sit from the mean.
  A small spread means steady months; a big spread means wild swings.

The `.describe()` method prints all of these at once.

In [ ]:
budget[["income", "spending", "saved"]].describe().round(1)

**Mean vs median &mdash; why keep both?** The mean gets pulled around by extreme
months (one huge birthday gift lifts the average). The median ignores the size of
the outliers and just reports the middle month. When they disagree, that gap is
itself a clue that a few unusual months are stretching the average.

In [ ]:
print(f"Average monthly income:  ${budget['income'].mean():.2f}")
print(f"Middle  monthly income:  ${budget['income'].median():.2f}")
print()
print(f"Average monthly spending: ${budget['spending'].mean():.2f}")
print(f"Average saved per month:  ${budget['saved'].mean():.2f}")
print(f"Months where you overspent: {int(budget['over_budget'].sum())} out of {len(budget)}")

### Step 3: See the shape of your spending

A single average hides the story. A **histogram** shows the whole shape: it sorts
months into bins and counts how many landed in each. Tall bars are common amounts;
lonely bars far to the right are the blow-out months.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(budget["spending"], bins=12, color="#4C72B0", edgecolor="white")
plt.axvline(budget["spending"].mean(), color="#C44E52", linestyle="--",
            label=f"mean ${budget['spending'].mean():.0f}")
plt.title("How much do you spend in a typical month?")
plt.xlabel("Spending in a month ($)")
plt.ylabel("Number of months")
plt.legend()
plt.show()

### Step 4: Where does the money actually go?

Now we split spending into its categories and take the average of each. A bar chart
sorts them so the biggest drains on your money sit at the top of your attention.

In [ ]:
categories = ["food", "transport", "entertainment", "clothes", "phone", "other"]
avg_by_category = budget[categories].mean().sort_values(ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(avg_by_category.index, avg_by_category.values, color="#55A868")
plt.title("Average spending per month, by category")
plt.xlabel("Average $ per month")
for i, v in enumerate(avg_by_category.values):
    plt.text(v + 0.5, i, f"${v:.0f}", va="center")
plt.show()

### Step 5: The calendar hides a pattern

Money is **seasonal**. Some months are predictably expensive. We pull the month
number out of the date and average spending across the same calendar month in every
year. Watch what happens in the summer and in December.

In [ ]:
# Turn "2024-07" into a real date, then grab the month number (1-12).
budget["date"] = pd.to_datetime(budget["month"])
budget["month_num"] = budget["date"].dt.month

by_month = budget.groupby("month_num")["spending"].mean()

plt.figure(figsize=(9, 4))
plt.plot(by_month.index, by_month.values, marker="o", color="#8172B3")
plt.title("Average spending by calendar month (all four years)")
plt.xlabel("Month of the year (1 = Jan, 12 = Dec)")
plt.ylabel("Average spending ($)")
plt.xticks(range(1, 13))
plt.grid(alpha=0.3)
plt.show()

The peaks are not random. Summer (months 6&ndash;8) brings more going out, and
December (month 12) brings holiday shopping. A pattern that repeats on the calendar
like this is called **seasonality**, and in Notebook 3 you will learn to measure it
properly.

### Step 6: Are you actually saving?

The **savings rate** is the share of income you keep: `saved / income`. Financial
advice often aims for around 20%. Let's see how this saver did, and watch the jar
fill up over four years.

In [ ]:
budget["savings_rate"] = budget["saved"] / budget["income"]
print(f"Average savings rate: {budget['savings_rate'].mean() * 100:.1f}% of income")

plt.figure(figsize=(9, 4))
plt.plot(budget["date"], budget["balance"], color="#4C72B0")
plt.fill_between(budget["date"], budget["balance"], alpha=0.2, color="#4C72B0")
plt.title("Savings jar balance over four years")
plt.xlabel("Date")
plt.ylabel("Balance ($)")
plt.grid(alpha=0.3)
plt.show()

### What you learned
- **Mean, median, and spread** are the first three numbers you reach for.
- A **histogram** shows the shape of the data, not just its centre.
- Spending is **seasonal** &mdash; the calendar predicts the blow-out months.
- The **savings rate** turns "am I doing okay?" into a number you can track.

### Try it yourself
- Change the histogram to show `income` instead of `spending`. Is it a wider or
  narrower shape? What does that tell you?
- Which single category would you cut to save the most? Prove it with the bar chart.

### Next
Notebook 2 uses these numbers to answer a real question: *if the future is
uncertain, what is the chance I hit my savings goal?* That is **Monte Carlo simulation**.